In [31]:
import os 
import pandas as pd 
import numpy as np
from dataclasses import dataclass
from pathlib import Path

In [3]:
pwd

'c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\Research'

In [4]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src')

In [5]:
@dataclass
class DataMetricsCalculationConfig:
    root_dir : Path 
    data_path : str 
    local_data_file : str

In [6]:
from ServicenowGamification.Utils.Common import * 
from ServicenowGamification.Constant import * 
from ServicenowGamification.Config import *

In [7]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification')

In [14]:
class ConfigurationManager:
    def __init__(self,
                 CONFIG_PATH=CONFIG_FILE_PATH,
                 PARAMS_PATH=PARAMS_FILE_PATH
                 ):
         print(CONFIG_PATH)
                 
         print(f"path config yaml {Path(CONFIG_PATH)}")
         self.config=Read_Yaml(Path(CONFIG_PATH))
         self.params=Read_Yaml(Path(PARAMS_PATH))
         self.Data_MetricsCalculation = self.config.Data_MetricsCalculation
    def get_metricscalculationconfig(self)->DataMetricsCalculationConfig:
         Config=self.Data_MetricsCalculation
         create_directories([Config.root_dir])
         data_preprocessingconfig=DataMetricsCalculationConfig(
              root_dir=Config.root_dir,
              data_path=Config.data_path,
              local_data_file=Config.local_data_file
         )
         return data_preprocessingconfig


In [39]:
class MetricsCalculation:
    def __init__(self,config:DataMetricsCalculationConfig):
        self.config=config
        self.params = Read_Yaml(PARAMS_FILE_PATH)
    def MTTR(self):
        df=pd.read_csv(self.config.data_path)
      
        #print(f"\n*-------------- Calculate MTTR(Mean Time To Resolve) ----------------*\n")
        date_columns = ["opened_at", "resolved_at", "closed_at"]
    
        for col in date_columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
        #print(df[date_columns].dtypes)    
            #Resolution Time = Resolved Time - Opened Time
            #df["resolution_time"] = (df["resolved_at"] - df["opened_at"])
        df["resolution_time"] = df["resolved_at"] - df["opened_at"]
            # Convert to hours:
            #df["resolution_hours"] = (df["resolution_time"].dt.total_seconds() / 3600)
            #df["resolution_hours"] = (df["resolution_time"].dt.total_seconds() / 3600)
        df["resolution_hours"] = ((df["resolved_at"] - df["opened_at"]).dt.total_seconds() / 3600)
           
        #print(f"\n*--------------  overall MTTR (Mean Time To Resolve) ----------------*\n")
           # print(df[["number", "resolution_hours"]].head())
        mttr = df["resolution_hours"].mean()
        print("MTTR:", mttr)
    
        #print(df[["number", "resolution_hours"]].head(1))   
        local_path=self.config.local_data_file
        #print(f"Preprocessed data saved to: {local_path}")
        #df.to_csv(local_path, index=False) 
        return df    

    def calculate_art(self,df):
        """
        Average Resolution Time grouped by assignment group.
        """
        df=df
        art = (

            df

            .groupby("assignment_group")["resolution_hours"]

            .mean()

            .round(2)

            .reset_index()

            .rename(
                columns={
                    "resolution_hours": "average_resolution_time"
                }
            )

        )

        print("\nAverage Resolution Time")

        print(art)

        return df,art


    def calculate_sla(self, df):
        df=df
        def get_status(row):
            priority = str(row["priority"])
                    #print(f"priority {priority}")
        
            if priority == "1 - Critical":
                return "Met" if row["resolution_hours"] <= self.params.sla.priority_1 else "Breached"
        
            elif priority == "2 - High":
                return "Met" if row["resolution_hours"] <= self.params.sla.priority_2 else "Breached"
        
            elif priority == "3 - Moderate":
                return "Met" if row["resolution_hours"] <= self.params.sla.priority_3 else "Breached"
        
            elif priority == "4 - Low":
                return "Met" if row["resolution_hours"] <= self.params.sla.priority_4 else "Breached"
        
            else:
                return "5 - Planning"

        df["sla_status"] = df.apply(
            get_status,
            axis=1
        )

        compliance = round(

            (
                df["sla_status"]
                .eq("Met")
                .mean()
            ) * 100,

            2

        )

        print(
            f"\nOverall SLA Compliance : {compliance}%"
        )

        return df
    def calculate_fcr(self,df):

        df["fcr"] = np.where(

            (
                (df["reopen_count"] == 0)
                &
                (df["reassignment_count"] == 0)
            ),

            1,

            0

        )

        fcr = round(

            df["fcr"].mean() * 100,

            2

        )

        print(f"\nFirst Call Resolution : {fcr}%")

        return df
    def performance_metrics(self,df):
        df=df
        total = len(df)

        closed = len(

            df[
                df["state"] == "Closed"
            ]

        )

        open_incidents = len(

            df[
                df["state"] != "Closed"
            ]

        )

        sla = round(

            (
                df["sla_status"]
                .eq("Met")
                .mean()
            ) * 100,

            2

        )

        mttr = round(

            df["resolution_hours"]
            .mean(),

            2

        )

        fcr = round(

            df["fcr"]
            .mean() * 100,

            2

        )

        metrics = {

            "Total Incidents": total,

            "Closed Incidents": closed,

            "Open Incidents": open_incidents,

            "MTTR(Hours)": mttr,

            "SLA Compliance(%)": sla,

            "FCR(%)": fcr

        }

        print("\nPerformance Metrics")

        for key, value in metrics.items():

            print(f"{key} : {value}")

        return df,metrics
    


    

In [40]:
config = ConfigurationManager()

metrics_calculation_config = config.get_metricscalculationconfig()

metrics_calculation = MetricsCalculation(metrics_calculation_config)   # ✅ Correct
df=metrics_calculation.MTTR()
df,art=metrics_calculation.calculate_art(df)
df=metrics_calculation.calculate_sla(df)
df=metrics_calculation.calculate_fcr(df)
df,metrics=metrics_calculation.performance_metrics(df)

Config\Config.yaml
path config yaml Config\Config.yaml
Reading YAML : Config\Config.yaml
[2026-08-02 11:30:22,397: INFO: Common: YAML file loaded successfully : Config\Config.yaml]
Reading YAML : params.yaml
[2026-08-02 11:30:22,405: INFO: Common: YAML file loaded successfully : params.yaml]
Received path : Artifacts/MetricsCalculation
Directory exists : True
Absolute path : C:\Users\RSR\PYTHON\ServicenowGamification\Artifacts\MetricsCalculation
[2026-08-02 11:30:22,413: INFO: Common: Created directory : Artifacts/MetricsCalculation]
Reading YAML : params.yaml
[2026-08-02 11:30:22,418: INFO: Common: YAML file loaded successfully : params.yaml]
MTTR: 742.1428824476651

Average Resolution Time
  assignment_group  average_resolution_time
0            Cloud                   882.00
1             Data                   798.84
2         Database                   792.80
3          Network                   462.65
4               QA                   826.67

Overall SLA Compliance : 11.59%

F